In [386]:
# VLM Evaluation
import os

def get_folder_names(path, folders=None):
    """
    Recursively finds and returns a list of the deepest folder paths
    that contain at least one JSON file.
    """
    if folders is None:
        folders = []

    for entry in os.listdir(path):
        full_path = os.path.join(path, entry)
        if os.path.isdir(full_path):
            # Check if this subfolder has any JSON files directly inside
            has_json = any(f.endswith(".json") for f in os.listdir(full_path) if os.path.isfile(os.path.join(full_path, f)))
            
            if has_json:
                folders.append(full_path)
            else:
                # Recurse deeper if no JSON files found here
                get_folder_names(full_path, folders)

    return folders


# Example usage:
folders = []
drone_sim_directory = "../results/drone_sim/"  # Replace with your actual folder path
images_sim_directory = "../results/images_sim/"  # Replace with your actual folder path
folders = get_folder_names(drone_sim_directory, folders)
folders = get_folder_names(images_sim_directory, folders)

# folders.remove('.archive')
# folders.remove('detections')

In [387]:
for folder in folders:
    print(folder)

../results/drone_sim/world2_scenario2_rescue/test10
../results/drone_sim/world2_scenario2_rescue/test9
../results/drone_sim/world2_scenario2_rescue/test7
../results/drone_sim/world2_scenario2_rescue/test6
../results/drone_sim/world2_scenario2_rescue/test1
../results/drone_sim/world2_scenario2_rescue/test8
../results/drone_sim/world2_scenario2_rescue/test4
../results/drone_sim/world2_scenario2_rescue/test3
../results/drone_sim/world2_scenario2_rescue/test2
../results/drone_sim/world2_scenario2_rescue/test5
../results/drone_sim/world3_scenario2_rescue/test10
../results/drone_sim/world3_scenario2_rescue/test9
../results/drone_sim/world3_scenario2_rescue/test7
../results/drone_sim/world3_scenario2_rescue/test6
../results/drone_sim/world3_scenario2_rescue/test1
../results/drone_sim/world3_scenario2_rescue/test8
../results/drone_sim/world3_scenario2_rescue/test4
../results/drone_sim/world3_scenario2_rescue/test3
../results/drone_sim/world3_scenario2_rescue/test2
../results/drone_sim/world3_s

In [388]:
import json
import re

In [389]:
def extract_assistance_section(text: str) -> str:
    """
    Return the '3. Assistance needed' section (case-insensitive) including its text.
    If not found, return an empty string.
    """
    text = text.replace("\\n", "\n")

    pattern = r"^\s*3\.\s*Assistance\s+needed\b[:\-–—]?\s*[\s\S]*?(?=(?:\n\s*\d+\.)|$)"
    match = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
    return match.group(0).strip() if match else None

In [390]:
results_data = []
vlm_responses = {}
llm_responses = {}
assist_inst_resp = {}

for folder in folders:
    path = folder.split('/')
    # print([path[-2][:16]])
    if 'world' == path[-2][:5] and 'scenario' == path[-2][7:15]:
        world_scenario = path[-2][:16]
        file_name = path[-2]+'_'+path[-1]
    else:
        world_scenario = path[-1]
        file_name = path[-1]

    results_file = f'{folder}/results.json'
    vlm_responses.update({world_scenario: []})
    llm_responses.update({world_scenario: []})
    assist_inst_resp.update({world_scenario: []})

    try:
        with open(results_file, 'r') as file:
            raw_data = json.load(file)

            # print(raw_data)

            if isinstance(raw_data, list) and raw_data and isinstance(raw_data[0], str):
                clean_data = [json.loads(item) for item in raw_data]
            else:
                clean_data = raw_data

            # print(clean_data)
        
        for result in clean_data:
            vlm_responses[world_scenario].append(result['vlm_description'])
            llm_responses[world_scenario].append(result['assistance_instructions'])
            
            asst_inst = extract_assistance_section(result['assistance_instructions'])
            # print(asst_inst)
            if asst_inst:
                assist_inst_resp[world_scenario].append(asst_inst)
            

        # print(results_data)
    except FileNotFoundError:
        print(f'{results_file} not found. Confirm file path or results file exsists.')
    except json.JSONDecodeError:
        print(f'{results_file} could not be decoded. Check file format.')
    except Exception as e:
        print(f'Unexcepted error {e}')

# # for resp in vlm_responses:
# #     for vlm_desc in vlm_responses[resp]:
# #         print(vlm_desc)


In [393]:
evaluation_references = 'evaluation_references.json'

try:
    with open(evaluation_references, 'r') as file:
        references = json.load(file)
    # print(results_data)
except FileNotFoundError:
    print(f'{evaluation_references} not found. Confirm file path or results file exsists.')

In [332]:
import sacrebleu

In [333]:
bleu_scores = {}
vlm_bleu = None
llm_bleu = None
asst_inst_bleu = None

for ref in references:
    world_and_scenario = ref['world_and_scenario']
    vlm_description_ref = ref['vlm_description']
    llm_description_ref = ref['llm_response']
    
    # print(world_and_scenario, vlm_description_ref, llm_description_ref, asst_inst_ref)
    # print(assist_inst_resp[world_and_scenario])

    vlm_bleu = sacrebleu.corpus_bleu(vlm_responses[world_and_scenario], vlm_description_ref)
    llm_bleu = sacrebleu.corpus_bleu(llm_responses[world_and_scenario], llm_description_ref)
    match = re.search(r"3\.\s*Assistance needed[\s\S]*?(?=\n\d+\.)", llm_description_ref)

    if match:
        asst_inst_ref = match.group().strip()
        asst_inst_bleu = sacrebleu.corpus_bleu(assist_inst_resp[world_and_scenario], asst_inst_ref)

    print(f'VLM BLEU Score: {vlm_bleu} \nLLM BLEU Score: {llm_bleu}\nAssistance Instructions BLEU Score: {asst_inst_bleu}')

VLM BLEU Score: BLEU = 0.81 2.2/1.1/0.6/0.3 (BP = 1.000 ratio = 45.000 hyp_len = 45 ref_len = 1) 
LLM BLEU Score: BLEU = 0.70 4.4/0.7/0.4/0.2 (BP = 1.000 ratio = 68.000 hyp_len = 68 ref_len = 1)
Assistance Instructions BLEU Score: None
VLM BLEU Score: BLEU = 0.97 4.4/1.1/0.6/0.3 (BP = 1.000 ratio = 45.000 hyp_len = 45 ref_len = 1) 
LLM BLEU Score: BLEU = 0.53 5.0/0.5/0.3/0.1 (BP = 1.000 ratio = 101.000 hyp_len = 101 ref_len = 1)
Assistance Instructions BLEU Score: None
VLM BLEU Score: BLEU = 0.95 4.3/1.1/0.6/0.3 (BP = 1.000 ratio = 46.000 hyp_len = 46 ref_len = 1) 
LLM BLEU Score: BLEU = 0.29 2.3/0.3/0.1/0.1 (BP = 1.000 ratio = 176.000 hyp_len = 176 ref_len = 1)
Assistance Instructions BLEU Score: None
VLM BLEU Score: BLEU = 0.97 4.4/1.1/0.6/0.3 (BP = 1.000 ratio = 45.000 hyp_len = 45 ref_len = 1) 
LLM BLEU Score: BLEU = 0.35 3.3/0.3/0.2/0.1 (BP = 1.000 ratio = 153.000 hyp_len = 153 ref_len = 1)
Assistance Instructions BLEU Score: None
VLM BLEU Score: BLEU = 0.95 4.3/1.1/0.6/0.3 (BP = 

In [368]:
for ref in references:
    world_and_scenario = ref['world_and_scenario']
    for i, cand in enumerate(assist_inst_resp[world_and_scenario]):
        print(f'\n{cand}\n')
    


3. Assistance Needed: The person is in the woods and unable to get up after falling, which suggests they may be injured or unable to move. First responders should bring medical supplies and equipment suitable for a wooded area rescue, such as stretchers and splints, to provide immediate medical attention and safely transport the individual to a hospital or medical facility. Additionally, ensure that the team has appropriate communication devices and lighting, as the area may be remote and poorly lit.


3. Assistance needed: The person appears to be in a park or wooded area and reported that they have fallen and cannot get up. Given that the caller stated they are unable to rise, it indicates they may be injured or in a compromising position. First responders should provide medical evaluation and physical assistance to safely help the person stand and assess any potential injuries. If the area is remote or the person's condition is severe, additional support such as carrying or stretch

In [420]:
from bert_score import score

# bert_score dict format:
# {
#   'world_and_scenario' : ''
#   , 'References' : {'VLM': '', LLM: ''}
#   , 'Candidates' : {'VLM': [], LLM: []}
#   , 'Bert_Scores' : {'VLM' : {'P': [], 'R': [], 'F1': []}, 'LLM' : {'P': [], 'R': [], 'F1': []}}
# }
bert_scores = []

# {
#   'world_and_scenario': ''
#   'LLM_Reference' : ''
#   'LLM_Candidate' : ''
#   'LLM_Precision' : ''
#   'LLM_Recall' : ''
#   'LLM_F1 Score' : ''
#   'Person_Found_match': ('True Positive', 'True Negative', 'False Positive', 'False Negative')
#   'Assistance_Required_Match' : ('True Positive', 'True Negative', 'False Positive', 'False Negative')
# }
LLM_bert_scores = []

# {
#   'world_and_scenario': ''
#   'VLM_Reference' : ''
#   'VLM_Candidate' : ''
#   'VLM_Precision' : ''
#   'VLM_Recall' : ''
#   'VLM_F1 Score' : ''
# }
VLM_bert_scores = []

asst_inst_bert_scores = []

vlm_bert = None
llm_bert = None
# asst_inst_bert = None
pattern = r"^\s*3\.\s*Assistance\s+needed\b[:\-–—]?\s*[\s\S]*?(?=(?:\n\s*\d+\.)|$)"

for ref in references:
    vlm_Precision_scores = []
    vlm_Recall_scores = []
    vlm_F1_scores = []

    llm_Precision_scores = []
    llm_Recall_scores = []
    llm_F1_scores = []

    asst_inst_Precision_scores = []
    asst_inst_Recall_scores = []
    asst_inst_F1_scores = []

    world_and_scenario = ref['world_and_scenario']
    vlm_ref = ref['vlm_description']
    llm_ref = ref['llm_response']
    asst_inst_ref = extract_assistance_section(ref['llm_response'])


    vlm_P, vlm_R, vlm_F1 = score(vlm_responses[world_and_scenario], [vlm_ref] * len(vlm_responses[world_and_scenario]), lang='en')
    llm_P, llm_R, llm_F1 = score(llm_responses[world_and_scenario], [llm_ref] * len(llm_responses[world_and_scenario]), lang='en')
    
    # vlm_bleu = sacrebleu.corpus_bleu(vlm_responses[world_and_scenario], vlm_description_ref)
    # llm_bleu = sacrebleu.corpus_bleu(llm_responses[world_and_scenario], llm_description_ref)    

    if assist_inst_resp[world_and_scenario]:
        match = re.search(pattern, ref['llm_response'], flags=re.IGNORECASE | re.MULTILINE)

        if match:
            asst_inst_ref = match.group(0).strip()
        else:
            asst_inst_ref = 'No Instructions.'

        asst_inst_P, asst_inst_R, asst_inst_F1 = score(assist_inst_resp[world_and_scenario], [asst_inst_ref] * len(assist_inst_resp[world_and_scenario]), lang='en')
        asst_inst_bleu = sacrebleu.corpus_bleu(assist_inst_resp[world_and_scenario], asst_inst_ref)    

        for i, cand in enumerate(assist_inst_resp[world_and_scenario]):
            asst_inst_Precision_scores.append(asst_inst_P[i].item())
            asst_inst_Recall_scores.append(asst_inst_R[i].item())
            asst_inst_F1_scores.append(asst_inst_F1[i].item())

            asst_inst_bert_scores.append({'world_and_scenario' : world_and_scenario
                                        , 'Assistance_Instruction_Reference': asst_inst_ref
                                        , 'Assistance_Instruction_Candidate': cand
                                        , 'Assistance_Instruction_BERT_Precision': asst_inst_P[i].item()
                                        , 'Assistance_Instruction_BERT_Recall': asst_inst_R[i].item()
                                        , 'Assistance_Instruction_BERT_F1_Score': asst_inst_F1[i].item()
                                        # , 'Assistance_Instruction_BLEU_Score': asst_inst_bleu
                                        })


    # print(asst_inst_P, asst_inst_R, asst_inst_F1)


    for i, cand in enumerate(vlm_responses[world_and_scenario]):
        vlm_Precision_scores.append(vlm_P[i].item())
        vlm_Recall_scores.append(vlm_R[i].item())
        vlm_F1_scores.append(vlm_F1[i].item())

        VLM_bert_scores.append({'world_scenario' : world_and_scenario
                                , 'VLM_Reference' : vlm_ref
                                , 'VLM_Candidate' : cand
                                , 'VLM_Precision' : vlm_P[i].item()
                                , 'VLM_Recall': vlm_R[i].item()
                                , 'VLM_F1_Score' : vlm_F1[i].item()
                                # , 'VLM_BLEU_Score': vlm_bleu
                                })
        
    for i, cand in enumerate(llm_responses[world_and_scenario]):
        llm_Precision_scores.append(llm_P[i].item())
        llm_Recall_scores.append(llm_R[i].item())
        llm_F1_scores.append(llm_F1[i].item())

        if '1. Person Found' in llm_ref and '1. Person Found' in cand:
            person_found_match = 'True Positive'
        elif '1. Person Found' in llm_ref and '1. No Person Found' in cand:
            person_found_match = 'False Negative'
        elif '1. No Person Found' in llm_ref and '1. No Person Found' in cand:
            person_found_match = 'True Negative'
        elif '1. No Person Found' in llm_ref and '1. Person Found' in cand:
            person_found_match = 'False Positive'

        if '2. Person Requires Immediate Assistance' in llm_ref and '2. Person Requires Immediate Assistance' in cand:
            assistance_required_match = 'True Positive'
        elif '2. Person Requires Immediate Assistance' in llm_ref and '2. Person does not require assistance' in cand:
            assistance_required_match = 'False Negative'
        elif '2. Person does not require assistance' in llm_ref and '2. Person does not require assistance' in cand:
            assistance_required_match = 'True Negative'
        elif '2. Person does not require assistance' in llm_ref and '2. Person Requires Immediate Assistance' in cand:
            assistance_required_match = 'False Positive'

        LLM_bert_scores.append({'world_scenario' : world_and_scenario
                                , 'LLM_Reference' : llm_ref
                                , 'LLM_Candidate' : cand
                                , 'LLM_Precision' : llm_P[i].item()
                                , 'LLM_Recall': llm_R[i].item()
                                , 'LLM_F1_Score' : llm_F1[i].item()
                                , 'Person_Found_Match': person_found_match
                                , 'Assistance_Required_Match': assistance_required_match
                                # , 'LLM_BLEU_Score': llm_bleu
                                })
        
    eval_bert_results = {'world_and_scenario': world_and_scenario
                         , 'References': {'VLM': vlm_ref, 'LLM': llm_ref}
                         , 'Candidates': {'VLM': vlm_responses[world_and_scenario], 'LLM': llm_responses[world_and_scenario]}
                         , 'Bert_Scores': {
                                        'VLM': {'Precision': vlm_Precision_scores, 'Recall': vlm_Recall_scores, 'F1': vlm_F1_scores}
                                        , 'LLM': {'Precision': llm_Precision_scores, 'Recall': llm_Recall_scores, 'F1': llm_F1_scores}
                                        , 'Assistance_Instructions': {'Precision': asst_inst_Precision_scores, 'Recall': asst_inst_Recall_scores, 'F1': asst_inst_F1_scores}
                                }
                        # , 'BLEU_Scores': {'VLM': vlm_bleu, 'LLM': llm_bleu, 'Assistance_Instruction': asst_inst_bleu}
                         }
    
    bert_scores.append(eval_bert_results)


/Users/davidlelis/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initializ

In [427]:
# Get BLEU Scores

for ref in references:
    world_and_scenario = ref['world_and_scenario']
    vlm_ref = ref['vlm_description'].split()
    llm_ref = ref['llm_response'].split()
    asst_inst_ref = extract_assistance_section(ref['llm_response'])

    if asst_inst_ref:
        asst_inst_ref = asst_inst_ref.split()

    print(asst_inst_ref)

None
['3.', 'Assistance', 'needed:', 'Person', 'is', 'lying', 'down', 'on', 'the', 'grass', 'of', 'the', 'woods.', 'The', 'person', 'may', 'be', 'unconscious', 'and', 'unresponsive.']
None
['3.', 'Assistance', 'needed:', 'There', 'are', 'two', 'people', 'at', 'the', 'scene.', 'One', 'of', 'them', 'is', 'lying', 'in', 'the', 'water', 'and', 'pay', 'be', 'injured', 'or', 'unconcious.', 'The', 'other', 'person', 'seems', 'safe.', 'Instruct', 'the', 'safe', 'person', 'to', 'care', 'for', 'the', 'injured/prone', 'person', 'until', 'help', 'arrives.']
None
['3.', 'Assistance', 'needed:', 'The', 'image', 'only', 'contains', 'one', 'person', 'and', 'the', 'message', 'indicates', 'the', 'other', 'driver', 'may', 'be', 'stuck', 'in', 'their', 'car.', 'Instruct', 'the', 'keep', 'the', 'scene', 'safe', 'and', 'wait', 'for', 'emergency', 'services.']
['3.', 'Assistance', 'needed:', 'The', 'individuals', 'in', 'the', 'image', 'seem', 'to', 'be', 'running', 'away', 'from', 'something', 'in', 'a', 'se

In [399]:
vlm_precision_scores = []
vlm_recall_scores = []
vlm_F1_scores = []
for score in bert_scores:
    for i in score['Bert_Scores']['VLM']['Precision']:
        vlm_precision_scores.append(i)
    for i in score['Bert_Scores']['VLM']['Recall']:
        vlm_recall_scores.append(i)
    for i in score['Bert_Scores']['VLM']['F1']:
        vlm_F1_scores.append(i)

avg_vlm_precision_scores = sum(vlm_precision_scores) / len(vlm_precision_scores)
avg_vlm_recall_scores = sum(vlm_recall_scores) / len(vlm_recall_scores)
avg_vlm_F1_scores = sum(vlm_F1_scores) / len(vlm_F1_scores)

print(avg_vlm_precision_scores, avg_vlm_recall_scores, avg_vlm_F1_scores)

0.8352789485354941 0.8941341685053366 0.8635195769307341


In [400]:
asst_inst_precision_scores = []
asst_inst_recall_scores = []
asst_inst_F1_scores = []
for score in bert_scores:
    for i in score['Bert_Scores']['Assistance_Instructions']['Precision']:
        asst_inst_precision_scores.append(i)
    for i in score['Bert_Scores']['Assistance_Instructions']['Recall']:
        asst_inst_recall_scores.append(i)
    for i in score['Bert_Scores']['Assistance_Instructions']['F1']:
        asst_inst_F1_scores.append(i)

avg_asst_inst_precision_scores = sum(asst_inst_precision_scores) / len(asst_inst_precision_scores)
avg_asst_inst_recall_scores = sum(asst_inst_recall_scores) / len(asst_inst_recall_scores)
avg_asst_inst_F1_scores = sum(asst_inst_F1_scores) / len(asst_inst_F1_scores)

print(avg_asst_inst_precision_scores, avg_asst_inst_recall_scores, avg_asst_inst_F1_scores)

ZeroDivisionError: division by zero

In [346]:
for score in VLM_bert_scores:
    print(score)

{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image is a  of a square with a white border. The square is rectangular in shape and appears to be made of a smooth, glossy material. The background is a solid black color. On the left side of the image', 'VLM_Precision': 0.8002822399139404, 'VLM_Recall': 0.8207932710647583, 'VLM_F1_Score': 0.8104079365730286}
{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image is a  of a person standing in front of a black background. The person is wearing a blue suit and has short dark hair. They are standing with their arms stretched out to the sides and their head tilted slightly to', 'VLM_Precision': 0.790613055229187, 'VLM_Recall': 0.8348599672317505, 'VLM_F1_Score': 0.8121342658996582}
{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image shows a  of a 

In [230]:
llm_precision_scores = []
llm_recall_scores = []
llm_F1_scores = []
for score in bert_scores:
    for i in score['Bert_Scores']['LLM']['Precision']:
        llm_precision_scores.append(i)
    for i in score['Bert_Scores']['LLM']['Recall']:
        llm_recall_scores.append(i)
    for i in score['Bert_Scores']['LLM']['F1']:
        llm_F1_scores.append(i)

avg_llm_precision_scores = sum(llm_precision_scores) / len(llm_precision_scores)
avg_llm_recall_scores = sum(llm_recall_scores) / len(llm_recall_scores)
avg_llm_F1_scores = sum(llm_F1_scores) / len(llm_F1_scores)

print(avg_llm_precision_scores, avg_llm_recall_scores, avg_llm_F1_scores)

0.8527964199093027 0.908042637730988 0.879291517465887


In [196]:
import pandas as pd

In [421]:
llm_bert_df = pd.DataFrame(LLM_bert_scores)
vlm_bert_df = pd.DataFrame(VLM_bert_scores)
asst_inst_bert_df = pd.DataFrame(asst_inst_bert_scores)

In [422]:
llm_bert_df.to_excel('LLM_Bert_Scores.xlsx', index=False)
vlm_bert_df.to_excel('VLM_Bert_Scores.xlsx', index=False)
asst_inst_bert_df.to_excel('Assistance_Instructions_Bert_Scores.xlsx', index=False)

In [199]:
# Precision, Recall, F1 Score
person_found_match_TP = 0
person_found_match_FP = 0
person_found_match_FN = 0
person_found_match_TN = 0

assistance_required_TP = 0
assistance_required_FP = 0
assistance_required_FN = 0
assistance_required_TN = 0

for item in LLM_bert_scores:
    # Person Found Match
    if item['Person_Found_Match'] == 'True Positive':
        person_found_match_TP += 1
    elif item['Person_Found_Match'] == 'False Positive':
        person_found_match_FP += 1
    elif item['Person_Found_Match'] == 'False Negative':
        person_found_match_FN += 1
    elif item['Person_Found_Match'] == 'True Negative':
        person_found_match_TN += 1

    if item['Assistance_Required_Match'] == 'True Positive':
        assistance_required_TP += 1
    elif item['Assistance_Required_Match'] == 'False Positive':
        assistance_required_FP += 1
    elif item['Assistance_Required_Match'] == 'False Negative':
        assistance_required_FN += 1
    elif item['Assistance_Required_Match'] == 'True Negative':
        assistance_required_TN += 1

In [200]:
# Preson Found Metrics
person_found_match_prec = person_found_match_TP / (person_found_match_TP + person_found_match_FP)
person_found_match_recall = person_found_match_TP / (person_found_match_TP + person_found_match_FN)
person_found_match_f1 = 2 * (person_found_match_prec*person_found_match_recall) / (person_found_match_prec+person_found_match_recall)

# Assistance Required Metrics
assistance_required_prec = assistance_required_TP / (assistance_required_TP + assistance_required_FP)
assistance_required_recall = assistance_required_TP / (assistance_required_TP + assistance_required_FN)
assistance_required_f1 = 2 * (assistance_required_prec*assistance_required_recall) / (assistance_required_prec+assistance_required_recall)

In [202]:
print(f'Person Found Precision: {person_found_match_prec}')
print(f'Person Found Recall: {person_found_match_recall}')
print(f'Person Found F1 Score: {person_found_match_f1}')
print(f'Assistance Required Precision: {assistance_required_prec}')
print(f'Assistance Required Recall: {assistance_required_recall}')
print(f'Assistance Required F1 Score: {assistance_required_f1}')

Person Found Precision: 1.0
Person Found Recall: 0.9183098591549296
Person Found F1 Score: 0.9574155653450807
Assistance Required Precision: 0.8093023255813954
Assistance Required Recall: 0.9405405405405406
Assistance Required F1 Score: 0.8699999999999999
